In [1]:
import pandas as pd
import bs4 as bs
import re
from group import RateGroup, RateAgreement, RateSteps
import json

In [2]:
classDf = pd.read_pickle('classDf.pkl')
idContentDf = pd.read_pickle('idContentDf.pkl')

In [10]:
def createJSON_CS_SP_EL_TC_TR_UT(tbsId, sep):
    id_3 = classDf.loc[classDf['ids'] == tbsId]
    ac = idContentDf.loc[idContentDf['ids'] == tbsId, 'htmlContent'].values[0]
    soup = bs.BeautifulSoup(ac,'lxml')

    rateTables = soup.select('h3:has(span[id^="rates"]) ~ table')
    rates = []
    for rateCat in rateTables:
        steps = rateCat.select('thead th')
        rateStpes = rateCat.select('tbody tr')
        rateAgreements = []
        for idx,agreement in enumerate(rateStpes):
            rateStepsList = []
            for idx2,amount in enumerate(agreement.findAll('td')):
                step = re.search('Step.[0-9]*', steps[idx2+1].getText())
                if amount.find(' to ') == -1:
                    newAmts = [amountgetText().replace(',', '')]
                else:
                    newAmts = amount.getText().replace(',', '').split(' to ')
                rateStep = RateSteps(
                    int(re.search('[0-9]',step[0])[0]),
                    [int(newAmt) for newAmt in newAmts]
                )
                rateStepsList.append(rateStep)
            rateAgreement = RateAgreement(
                agreement.find('time').get('datetime'),
                rateStepsList
            )
            rateAgreements.append(rateAgreement)
        rateGrp = RateGroup(
            rateCat.find('caption').getText().split(sep)[0].strip(),
            rateCat.find('caption').getText().split(sep)[0].rsplit('-',1)[0].strip(),
            rateCat.find('caption').getText().split(sep)[0].rsplit('-',1)[1].strip().lstrip('0'),       
            rateAgreements
        )
        rates.append(rateGrp)

    results = [rateGrp.to_dict() for rateGrp in rates]
    with open('data/' + tbsId + '.json', 'w') as json_file:
        json.dump(results, json_file)
    return results

In [11]:
page_3 = createJSON_CS_SP_EL_TC_TR_UT('3', ':')
page_9 = createJSON_CS_SP_EL_TC_TR_UT('9', ':')
page_25 = createJSON_CS_SP_EL_TC_TR_UT('25', ':')
page_26 = createJSON_CS_SP_EL_TC_TR_UT('26', ' - ')
page_27 = createJSON_CS_SP_EL_TC_TR_UT('27', ':')

combined_list = page_3 + page_9 + page_25 + page_26 + page_27

In [12]:
with open('data/combined/payscales.json', 'w') as json_file:
        json.dump(combined_list, json_file)